In [1]:
from pyiceberg.catalog import load_catalog
from pyiceberg.schema import Schema
from pyiceberg.types import NestedField, LongType, StringType, DoubleType
import pyarrow as pa
from pprint import pprint

In [2]:
catalog = load_catalog("default")
print(catalog.list_namespaces())

[]


In [3]:
# Create a namespace and table

catalog.create_namespace("demo")
print(catalog.list_namespaces())

[('demo',)]


In [4]:
schema = Schema(
    NestedField(1, "id",       LongType(),    required=True),
    NestedField(2, "name",     StringType()),
    NestedField(3, "category", StringType()),
    NestedField(4, "amount",   DoubleType()),
)
table = catalog.create_table("demo.orders", schema=schema)
print(table)

orders(
  1: id: required long,
  2: name: optional string,
  3: category: optional string,
  4: amount: optional double
),
partition by: [],
sort order: [],
snapshot: null


In [5]:
# add some data 
batch = pa.table({
    "id":       pa.array([1, 2, 3], pa.int64()),
    "name":     pa.array(["Alice", "Bob", "Carol"], pa.string()),
    "category": pa.array(["A", "B", "A"], pa.string()),
    "amount":   pa.array([100.0, 200.0, 150.0], pa.float64()),
},
schema=pa.schema([
        pa.field("id",       pa.int64(),  nullable=False),  # matches required
        pa.field("name",     pa.string(), nullable=True),
        pa.field("category", pa.string(), nullable=True),
        pa.field("amount",   pa.float64(),nullable=True),
    ])
)

table.append(batch)

In [6]:
import boto3, json

s3 = boto3.client(
    "s3",
    endpoint_url="http://minio:9000",
    aws_access_key_id="minioadmin",
    aws_secret_access_key="minioadmin",
)

# List everything under the table prefix
response = s3.list_objects_v2(Bucket="warehouse")

for obj in response.get("Contents", []):
    print(obj["Key"])


demo/orders/data/00000-0-0445b4f7-274c-4fa0-bb7a-345418a41147.parquet
demo/orders/metadata/00000-82a787f0-a8d7-454d-b19e-9e6d85efc024.metadata.json
demo/orders/metadata/00001-9e9447d6-da14-4d2b-abd3-7cf80b443449.metadata.json
demo/orders/metadata/0445b4f7-274c-4fa0-bb7a-345418a41147-m0.avro
demo/orders/metadata/snap-6105448148631833964-0-0445b4f7-274c-4fa0-bb7a-345418a41147.avro


In [7]:
meta_files = [
    o["Key"] for o in response["Contents"]
    if o["Key"].endswith(".metadata.json")
]
latest = sorted(meta_files)[-1]

In [9]:
obj = s3.get_object(Bucket="warehouse", Key=latest)
meta = json.loads(obj["Body"].read())
print("Format version:  ", meta["format-version"])
print("Table UUID:      ", meta["table-uuid"])
print("Location:        ", meta["location"])
print("Current snapshot:", meta["current-snapshot-id"])
print("\nSnapshots:")
for snap in meta["snapshots"]:
    print(f"  {snap['snapshot-id']}  →  manifest-list: {snap['manifest-list']}")


Format version:   2
Table UUID:       a34bc1f8-47c9-475f-803c-485cca72f118
Location:         s3://warehouse/demo/orders
Current snapshot: 6105448148631833964

Snapshots:
  6105448148631833964  →  manifest-list: s3://warehouse/demo/orders/metadata/snap-6105448148631833964-0-0445b4f7-274c-4fa0-bb7a-345418a41147.avro


In [10]:
# how the metadata file looks like !!
pprint(meta)

{'current-schema-id': 0,
 'current-snapshot-id': 6105448148631833964,
 'default-sort-order-id': 0,
 'default-spec-id': 0,
 'format-version': 2,
 'last-column-id': 4,
 'last-partition-id': 999,
 'last-sequence-number': 1,
 'last-updated-ms': 1780951521480,
 'location': 's3://warehouse/demo/orders',
 'metadata-log': [{'metadata-file': 's3://warehouse/demo/orders/metadata/00000-82a787f0-a8d7-454d-b19e-9e6d85efc024.metadata.json',
                   'timestamp-ms': 1780951518004}],
 'partition-specs': [{'fields': [], 'spec-id': 0}],
 'partition-statistics': [],
 'properties': {'write.parquet.compression-codec': 'zstd'},
 'refs': {'main': {'snapshot-id': 6105448148631833964, 'type': 'branch'}},
 'schemas': [{'fields': [{'id': 1,
                          'name': 'id',
                          'required': True,
                          'type': 'long'},
                         {'id': 2,
                          'name': 'name',
                          'required': False,
                 

In [11]:
import subprocess
subprocess.run(["pip", "install", "fastavro", "-q"])

CompletedProcess(args=['pip', 'install', 'fastavro', '-q'], returncode=0)

In [12]:
# how manifest list looks like
import fastavro, io

ml_key = meta["snapshots"][-1]["manifest-list"].replace("s3://warehouse/", "")

avro_bytes = s3.get_object(Bucket="warehouse", Key=ml_key)["Body"].read()
reader = fastavro.reader(io.BytesIO(avro_bytes))

print("Manifest list entries:")
for record in reader:
    print(f"  path:          {record['manifest_path']}")
    print(f"  added files:   {record['added_files_count']}")
    print(f"  existing files:{record['existing_files_count']}")
    print(f"  deleted files: {record['deleted_files_count']}")

Manifest list entries:
  path:          s3://warehouse/demo/orders/metadata/0445b4f7-274c-4fa0-bb7a-345418a41147-m0.avro
  added files:   1
  existing files:0
  deleted files: 0


In [13]:
# inspection using PyIceberg
# Current snapshot
snap = table.current_snapshot()
print(snap)

# All snapshots (time travel history)
for s in table.history():
    print(s)

# Files in the current snapshot
for f in table.scan().plan_files():
    print(f.file.file_path, f.file.record_count)


Operation.APPEND: id=6105448148631833964, schema_id=0
snapshot_id=6105448148631833964 timestamp_ms=1780951521463
s3://warehouse/demo/orders/data/00000-0-0445b4f7-274c-4fa0-bb7a-345418a41147.parquet 3


In [14]:
# Iceberg tracks every write as a snapshot. You can read any historical state by snapshot ID or timestamp.
for snap in table.history():
    print(snap.snapshot_id, snap.timestamp_ms)

history = table.history()
first_snapshot_id = history[0].snapshot_id

first_snapshot = table.scan(snapshot_id = first_snapshot_id).to_arrow().to_pandas()
print(first_snapshot)

6105448148631833964 1780951521463
   id   name category  amount
0   1  Alice        A   100.0
1   2    Bob        B   200.0
2   3  Carol        A   150.0


In [15]:
batch_2 = pa.table({
    "id":       pa.array([4, 5], pa.int64()),
    "name":     pa.array(["Dave", "Eve"], pa.string()),
    "category": pa.array(["B", "C"], pa.string()),
    "amount":   pa.array([300.0, 50.0], pa.float64()),
},
schema=pa.schema([
        pa.field("id",       pa.int64(),  nullable=False),
        pa.field("name",     pa.string(), nullable=True),
        pa.field("category", pa.string(), nullable=True),
        pa.field("amount",   pa.float64(),nullable=True),
    ])
)
table.append(batch_2)

In [16]:
table = catalog.load_table("demo.orders")
history = table.history()
print(f"Total snapshots: {len(history)}")
print(history)

Total snapshots: 2
[SnapshotLogEntry(snapshot_id=6105448148631833964, timestamp_ms=1780951521463), SnapshotLogEntry(snapshot_id=6090357139376724719, timestamp_ms=1780951564726)]


In [17]:
for snap in table.snapshots():
    print(f"\nSnapshot ID : {snap.snapshot_id}")
    print(f"Timestamp   : {snap.timestamp_ms}")
    print(f"Operation   : {snap.summary}")
    print(f"Manifest    : {snap.manifest_list}")


Snapshot ID : 6105448148631833964
Timestamp   : 1780951521463
Operation   : operation=Operation.APPEND
Manifest    : s3://warehouse/demo/orders/metadata/snap-6105448148631833964-0-0445b4f7-274c-4fa0-bb7a-345418a41147.avro

Snapshot ID : 6090357139376724719
Timestamp   : 1780951564726
Operation   : operation=Operation.APPEND
Manifest    : s3://warehouse/demo/orders/metadata/snap-6090357139376724719-0-048be0c5-2356-4d60-857d-0e2d8837138f.avro


In [18]:
for snap in table.snapshots():
    files = list(table.scan(snapshot_id=snap.snapshot_id).plan_files())
    print(f"Snapshot {snap.snapshot_id} → {len(files)} file(s)")
    
    for f in files:
        print(f"  {f.file.file_path}  rows={f.file.record_count}")

Snapshot 6105448148631833964 → 1 file(s)
  s3://warehouse/demo/orders/data/00000-0-0445b4f7-274c-4fa0-bb7a-345418a41147.parquet  rows=3
Snapshot 6090357139376724719 → 2 file(s)
  s3://warehouse/demo/orders/data/00000-0-048be0c5-2356-4d60-857d-0e2d8837138f.parquet  rows=2
  s3://warehouse/demo/orders/data/00000-0-0445b4f7-274c-4fa0-bb7a-345418a41147.parquet  rows=3
